# Machine Learning Pipelines

You are given a partly written notebook. You finish it, so that a machine-learning pipeline runs
from the raw accounting figures of a listed company through to a recommendation a lender could act
on. The task is a real one. From what a company's accounts said at the end of a fiscal year, predict
whether it filed for bankruptcy in the following year.

## The submission contract

This notebook is marked by a script that runs it from a clean start and reads the variables you
produce. Three rules make that possible, and breaking any of them costs marks.

1. **A cell marked `[LOCKED]` is not yours to change.** It holds the settings, the seeds and the cost
   figures every submission shares. Changing one makes your results incomparable with everyone
   else's, and the script notices.
2. **A cell marked `[YOU COMPLETE THIS]` must produce the variables its heading names**, spelled
   exactly as written. The script looks for those names. A correct answer under a different name
   earns nothing, because nothing is reading it.
3. **Where a step asks for a comment, write one**, in the cell that asks for it. A comment says
   why the line is there, not what the line does. `# fill the gaps with the median` is what.
   `# the median is used rather than the mean because a few very large firms would drag a mean
   upward` is why.

You may add cells. You may not delete the ones already here, and you may not change the
first line of any of them: the marking script finds each cell by that line.

## Before you start

- Put your dataset file in the same folder as this notebook. It is named `version_0.csv` through
  `version_9.csv`, and yours is the one matching the last digit of your student number.
- Run **Kernel, then Restart and Run All** before you submit, and watch it finish. A notebook that
  stops partway loses the marks for every step after the stop.
- Use AI assistance freely. You are responsible for everything in the file, and the same material is
  examined in person, without AI, in the invigilated term test.

In [46]:
# [LOCKED] Libraries. Do not change anything in this cell.
import hashlib

import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("scikit-learn", sklearn.__version__)

pandas 3.0.5
numpy 2.5.3
scikit-learn 1.9.1


## Step 0. Who you are, and which version you are working on

Fill in all three. `VERSION` is **the last digit of your student number**, as a whole number. If your
student number ends in 7, your dataset is `version_7.csv` and `VERSION` is `7`.

In [47]:
# [YOU COMPLETE THIS] Your identity and your dataset version.

STUDENT_NAME = "Martin Ramirez"      # your full name, spelled as it is on the class list
STUDENT_NUMBER = "001223661"    # your student number, digits only
VERSION = 1         # the last digit of your student number, as a whole number

In [48]:
# [LOCKED] Settings. Do not change anything in this cell.
RANDOM_STATE = 42
TRAIN_END_YEAR = 2014

# What each kind of mistake costs the lender, in dollars.
COST_MISSED_FAILURE = 25_000_000   # a company fails and the model did not flag it
COST_FALSE_ALARM = 250_000         # a healthy company is flagged and the business is refused

THRESHOLD_A = 0.50
THRESHOLD_B = 0.70

TRAP_COLUMN = "going_concern_flag"
RATIO_COLUMNS = [
    "working_capital_to_assets",
    "retained_earnings_to_assets",
    "ebit_to_assets",
    "market_value_to_liabilities",
    "sales_to_assets",
    "return_on_assets",
    "current_ratio",
    "leverage",
]

DATA_FILE = "version_" + str(VERSION) + ".csv"
np.random.seed(RANDOM_STATE)
print("reading", DATA_FILE)

reading version_1.csv


## Step 1. Load the data and take its measure

Read your file, look at it, and record four things. Three of them are the **fingerprint** of your
version: the number of rows, the number of missing values in each column, and a hash of the set of
companies in your file. The marking script recomputes all three and checks them against the version
you declared in Step 0, which is how it confirms you worked on the data you say you worked on. It
reports separately whether that version is the one your student number gives you.

Produce: `df`, `n_rows`, `n_cols`, `missing_counts`, `company_set_hash`. Print `df.head()`.

**Comment required.** Say what the rows of this file are. One row is not one company.

In [49]:
# [YOU COMPLETE THIS] Step 1. Load the data and take its measure.

df = pd.read_csv("version_1.csv")

n_rows = df.shape[0]       # how many rows the file has
n_cols = df.shape[1]       # how many columns the file has
missing_counts = df.isna().sum() #- one count per column, as a pandas Series

# The fingerprint of the company set. This line is written for you.
company_set_hash = hashlib.sha256(
    ",".join(str(i) for i in sorted(df["company_id"].unique())).encode("utf-8")
).hexdigest()

# Each row is one company's fiscal year. There are multiple entries per firm across years.
# print(df.head())

# Print the four values above so you can read them.

# print("rows:", n_rows)
# print("cols:", n_cols)
# print("company hash:", company_set_hash)
print(missing_counts)

company_id                          0
year                                0
sector                              0
current_assets                      0
total_assets                        0
cost_of_goods_sold                  0
total_long_term_debt                0
depreciation_and_amortization       0
ebit                                0
ebitda                              0
gross_profit                        0
inventory                         970
total_current_liabilities           0
net_income                          0
retained_earnings                   0
total_receivables                 647
total_revenue                       0
market_value                     1294
total_liabilities                   0
net_sales                           0
total_operating_expenses            0
going_concern_flag                  0
target                              0
dtype: int64


## Step 2. The column that arrived as text

`market_value` is stored as text, and the same amount is written three ways: `$1,250,000`,
`1,250,000`, and `1250000`. Some cells are blank. A model cannot read any of it until it is a number.

Count how many cells are not blank and still will not read as a number as they stand, then
convert the whole column to numbers. A blank must stay missing. It must not become a zero, because a company whose market value
was not recorded is not a company worth nothing.

Produce: `market_value_unreadable` (a count), and `market_value_median` (the median of the column
after you have converted it). Leave the converted numbers in `df["market_value"]`.

**Comment required.** Say why a blank must not be turned into a zero here.

In [50]:
# [YOU COMPLETE THIS] Step 2. Turn the money text into numbers.

raw_market_value = df["market_value"]

# A cell is unreadable when it is not blank and still will not read as a number.
as_number = pd.to_numeric(raw_market_value, errors="coerce")
market_value_unreadable = int((raw_market_value.notna() & as_number.isna()).sum())

# Strip the money punctuation, then convert. Write the result back into df["market_value"].
df["market_value"] = pd.to_numeric(
    raw_market_value.str.replace(r"[$,]", "", regex=True)
)

market_value_median = df["market_value"].median()

# Your comment here: Adding a value where none was recorded would skew the information, dragging the median down along with any other summary statistics. Adding a zero here would suggest the company is worthless.
# Print both values.

print("unreadable:", market_value_unreadable)
print("median:", market_value_median)

unreadable: 31507
median: 255598900.0


## Step 3. Check that the cleaning did not cost you data

Cleaning a column is the easiest place in a pipeline to destroy data without noticing. A conversion
that quietly fails turns readable money into missing values, and nothing in the code complains.

Count the missing values again, column by column, and compare with what you found in Step 1. The
counts for `market_value` should be the same as before, because you converted the unreadable cells
rather than discarding them. Then record, in one sentence, when and where the gaps that really are
missing get filled.

Keep every row. The gaps are filled with the median inside the model pipeline at Step 8, after the
split, and not here. Throwing away the rows that carry a gap changes every count, every ratio and
every figure you report from this point on.

Produce: `missing_after_clean`, a count of missing values for every column as in Step 1, and
`MISSING_STRATEGY` as a sentence in a string.

**Comment required.** Say what it would have meant if the missing count for `market_value` had gone
up at this step.

In [51]:
# [YOU COMPLETE THIS] Step 3. Audit the missing values after cleaning.

missing_after_clean = df.isna().sum()   # the same again, as a pandas Series

# Your comment here: what would a rise in the missing count have meant?

# A rise in missing count would indicate that during the conversion market values were converted in NaN values, in turn, these rows would later be filled in step 8, replacing real numbers with the median.

MISSING_STRATEGY = "The gaps are filled with the median inside the model pipeline at Step 8, after the train/test split, so the fill values are learned from the training years only."   # one sentence: where do the gaps get filled, and at which step?

# Print both.

print (missing_after_clean)

print (MISSING_STRATEGY)

company_id                          0
year                                0
sector                              0
current_assets                      0
total_assets                        0
cost_of_goods_sold                  0
total_long_term_debt                0
depreciation_and_amortization       0
ebit                                0
ebitda                              0
gross_profit                        0
inventory                         970
total_current_liabilities           0
net_income                          0
retained_earnings                   0
total_receivables                 647
total_revenue                       0
market_value                     1294
total_liabilities                   0
net_sales                           0
total_operating_expenses            0
going_concern_flag                  0
target                              0
dtype: int64
The gaps are filled with the median inside the model pipeline at Step 8, after the train/test split, so the fill 

## Step 4. The leakage check

**Leakage** is accidentally letting the model peek at the answer. One column in this file is a trap.
`going_concern_flag` is stamped on the file **after** the fiscal year has closed, once the company's
outcome for the following year is known. It is not something a lender could look up on the day the
decision has to be made.

Do not take that on trust. Test a suspect column the way a suspect column is always tested:
cross-tabulate it against the target, and compare the failure rate among flagged firms with the rate
among the rest. Then act on what you find.

Produce: `leak_table` (a cross-tabulation of `going_concern_flag` against `target`),
`fail_rate_with_flag`, `fail_rate_without_flag`.

**Comment required.** State the timing question you would ask of any column before using it, and the
answer this column gives.

In [52]:
# [YOU COMPLETE THIS] Step 4. Test the suspect column before deciding anything about it.

leak_table = pd.crosstab(df["going_concern_flag"], df["target"])   # a 2 by 2 pandas cross-tabulation of the flag against the target

fail_rate_with_flag = leak_table.loc[1, 1] / leak_table.loc[1].sum()      # share of flagged firm-years that failed
fail_rate_without_flag = leak_table.loc[0, 1] / leak_table.loc[0].sum()   # share of unflagged firm-years that failed

# Your comment here:

# Is the value known at the point of decision, in this case when we are providing a loan?. This column is stamped after the outcome is known, flagged firm-years
# fail about 42% of the time against under 0.1% for the rest.


# Print the table and both rates.

print(leak_table)
print(round(fail_rate_with_flag, 5))
print (round(fail_rate_without_flag, 5))

target                  0    1
going_concern_flag            
0                   63541   36
1                     641  468
0.422
0.00057


## Step 5. Build the ratios

The file gives you raw dollars, not ratios. Turning statement lines into ratios is the analyst's
work, and it is what makes one company comparable with another: a $50 MM loss means one thing
at a corner shop and another at an airline.

Compute these eight columns, spelled exactly as listed, on every row. Do not clip, winsorize or
round the results. The marking script compares each column's count of values, its total, its
smallest value and its largest value with the reference, so a trimmed tail or a rounded column
reads as a different column. Five of them are the ratios behind the
classic distress models of corporate finance, and three are the plain measures any credit file
carries.

| Column | Formula |
|---|---|
| `working_capital_to_assets` | (current assets less total current liabilities) / total assets |
| `retained_earnings_to_assets` | retained earnings / total assets |
| `ebit_to_assets` | EBIT / total assets |
| `market_value_to_liabilities` | market value / total liabilities |
| `sales_to_assets` | net sales / total assets |
| `return_on_assets` | net income / total assets |
| `current_ratio` | current assets / total current liabilities |
| `leverage` | total liabilities / total assets |

**Comment required.** Say why a ratio tells you more about distress than the dollar figure it was
built from.

In [ ]:
# [YOU COMPLETE THIS] Step 5. Turn the statement lines into the eight named ratios.

df["working_capital_to_assets"] = (
    df["current_assets"] - df["total_current_liabilities"]
) / df["total_assets"]
df["retained_earnings_to_assets"] = df["retained_earnings"] / df["total_assets"]
df["ebit_to_assets"] = df["ebit"] / df["total_assets"]

# The remaining five, following the pattern above and the table in the heading.
df["market_value_to_liabilities"] = df["market_value"] / df["total_liabilities"]
df["sales_to_assets"] = df["net_sales"] / df["total_assets"]
df["return_on_assets"] = df["net_income"] / df["total_assets"]
df["current_ratio"] = df["current_assets"] / df["total_current_liabilities"]
df["leverage"] = df["total_liabilities"] / df["total_assets"]

# Your comment here: why does a ratio say more about distress than the dollar figure?

# A ratio allows us to conduct an apples to apples comparison. The dollar figure only means something relative to the size of the company. When we look at the ratios we are able compare firms of any scale.

# Print a summary of the eight ratio columns.
print(df[RATIO_COLUMNS].describe())

       working_capital_to_assets  retained_earnings_to_assets  ebit_to_assets  \
count               64686.000000                 64686.000000    64686.000000   
mean                    0.194511                    -1.572841       -0.070892   
std                     0.708299                     7.083189        0.556517   
min                   -63.206107                  -465.681010      -34.129072   
25%                     0.047499                    -0.859729       -0.065796   
50%                     0.218041                     0.002292        0.050684   
75%                     0.428992                     0.278061        0.106461   
max                     0.992889                    24.039409        6.475407   

       market_value_to_liabilities  sales_to_assets  return_on_assets  \
count                 63392.000000     64686.000000      64686.000000   
mean                      8.173891         1.095314         -0.147812   
std                      91.495001         0.970834

## Step 6. Features and target

Choose what the model is allowed to see. The eight ratios are the features. Everything else stays
out, and two exclusions matter enough to name.

`going_concern_flag` stays out, for the reason you established in Step 4. The raw dollar columns stay
out because the ratios already carry what they say, on a scale that compares one company with
another. `company_id`, `year` and `sector` stay out too. They are not measurements of the
business, and there is a sharper reason besides.

A failure is recorded on the **last fiscal year a company appears in**. Anything built out of
where a row sits in a company's record, how long that record runs, or whether it is the last one,
carries the same defect `going_concern_flag` does. It is information from after the decision.
Standing at the end of a fiscal year, nobody knows whether the company will report again. A feature like that will make your model look far better than it is, on this data and
nowhere else. Leave the identifiers out, and do not build anything from them.

Produce: `X` (the eight ratio columns, in the order `RATIO_COLUMNS` gives) and `y` (the target).

**Keep `X` as a pandas DataFrame with the eight columns named.** The marking script reads the
column names to confirm the trap column and the target are not among your features. Converting it
to a plain array throws the names away, and those points with them.

**Comment required.** Name the column you left out on purpose, and why.

In [ ]:
# [YOU COMPLETE THIS] Step 6. Decide what the model may see.

X = df[RATIO_COLUMNS]   # the eight ratio columns, in the order RATIO_COLUMNS gives
y = df["target"]   # the target

# Your comment here: which column did you leave out on purpose, and why?

# We would leave the going concern flag column out. In an earlier step we validated that those firms marked with this flag failed 42% of the time as compared to less than 1%, as such this would spoil the model.
# Print the feature names, the number of rows, and the number of failures.

print(list(X.columns))
print("rows:", len(X))
print("failures:", int(y.sum()))

['working_capital_to_assets', 'retained_earnings_to_assets', 'ebit_to_assets', 'market_value_to_liabilities', 'sales_to_assets', 'return_on_assets', 'current_ratio', 'leverage']
rows: 64686
failures: 504


## Step 7. Split by time, not at random

This is a panel: one company appears in the file many times, once for each year it reported. A
company that fails contributes several surviving years before its failing one.

Split those rows at random and the same company lands in both halves. The model then recognizes the
company rather than forecasting it, and the score you report is flattering and false. It is also not
the question a lender asks, which is always about a year that has not happened yet.

So train on the earlier years and test on the later ones. Train on every fiscal year up to and
including `TRAIN_END_YEAR`, and test on every year after it.

Produce: `X_train`, `X_test`, `y_train`, `y_test`, and `train_years` and `test_years` as sorted
lists of the fiscal years that actually appear in each half, read from the rows you put there.

**Keep the four of them as pandas objects, with the row labels they came with.** Later steps read
those labels, and so does the marking script, which uses them to confirm that the rows in your
test set really are from later years. Converting them to plain arrays, or resetting the index,
throws that away.

**Comment required.** Say what a random split would have let the model see that a time split does
not.

In [ ]:
# [YOU COMPLETE THIS] Step 7. Split on the calendar, not at random.

train_mask = df["year"] <= TRAIN_END_YEAR   # the rows from TRAIN_END_YEAR and earlier
test_mask = df["year"] > TRAIN_END_YEAR    # the rows after TRAIN_END_YEAR

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
train_years = sorted(df.loc[train_mask, "year"].unique())   # sorted list of the years in the training half
test_years = sorted(df.loc[test_mask, "year"].unique())    # sorted list of the years in the test half

# Your comment here: what would a random split let the model see?

# A random split would allow the model to peek at future outcomes, leaning not from patterns but outcomes resultant from the company being put in both splits.

print("train:", X_train.shape, train_years[0], "to", train_years[-1], "failures:", int(y_train.sum()))
print("test:", X_test.shape, test_years[0], "to", test_years[-1], "failures:", int(y_test.sum()))

train: (54680, 8) 1999 to 2014 failures: 406
test: (10006, 8) 2015 to 2018 failures: 98


## Step 8. Fit the model on the training years

Build a pipeline with three steps in this order: a `SimpleImputer` with `strategy="median"`, a
`StandardScaler`, then a `LogisticRegression`. Fit it on the training half only. Those three, in
that order, are what your figures are compared against; another imputer or another scaler reaches
a similar place by a different route and gives different numbers everywhere after this step.

The order matters and so does the container. A pipeline learns its fill values and its scaling from
whatever it is fitted to. Fit it after the split and it learns them from the training years alone,
which is the honest arrangement you described in Step 3. Fill the gaps yourself before the split and
the test years have already shaped the numbers the model was trained on.

Give the logistic regression `max_iter=5000` and `random_state=RANDOM_STATE`. Your figures are
compared with a reference run that used both. A model left to stop at the default number of steps
has not finished converging, and it gives different coefficients and different counts.

Produce: `model_raw`, fitted.

**Comment required.** Say why the filling belongs inside the pipeline rather than before the split.

In [67]:
# [YOU COMPLETE THIS] Step 8. Fit the pipeline on the raw, unbalanced training years.

model_raw = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])   # a Pipeline of the three steps named above, with max_iter=5000
                   # and random_state=RANDOM_STATE on the logistic regression
# Fit it on the training half only.

model_raw.fit(X_train, y_train)

# Your comment here: why does the filling belong inside the pipeline?
# If we were to fill before the pipeline was run the model would learn from the whole data set rather than just the training set which in turn would shape its predictions. 
# Print the fitted pipeline.

print(model_raw)

Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
                ('model', LogisticRegression(max_iter=5000, random_state=42))])


## Step 9. Score the raw model on the held-out years, and report it in full

Score the test years once. Report the accuracy beside the **majority baseline**, which is the
accuracy of a rule that predicts survival for every company and fits no model at all. Report the
recall beside it, which is the share of the companies that actually failed that the model flagged.

Use `THRESHOLD_A` to turn probabilities into predictions. If your model flags no company at all,
precision is being asked for a share of an empty set. Report it as zero rather than letting the
division fail.

Produce: `prob_raw`, `pred_raw`, `raw_tn`, `raw_fp`, `raw_fn`, `raw_tp`, `raw_accuracy`,
`raw_precision`, `raw_recall`, `raw_f1`, `baseline_accuracy`.

**Comment required.** Put the accuracy and the recall side by side in words, and say what you
reported for precision if nothing was flagged.

In [ ]:
# [YOU COMPLETE THIS] Step 9. Score the raw model on the years it has not seen.

prob_raw = model_raw.predict_proba(X_test)[:, 1]   # the predicted probability of failure for each test row, kept
                  # unrounded: rounded copies collapse to too few distinct values
pred_raw = (prob_raw >= THRESHOLD_A).astype(int)   # 1 where that probability reaches THRESHOLD_A, else 0, in the
                  # same order as the test rows

raw_tn, raw_fp, raw_fn, raw_tp = confusion_matrix(y_test, pred_raw).ravel()

raw_accuracy = (raw_tp + raw_tn) / len(y_test)
raw_precision = raw_tp / (raw_tp + raw_fp) if (raw_tp + raw_fp) > 0 else 0.       # report 0.0 where nothing was flagged
raw_recall = raw_tp / (raw_tp + raw_fn)
raw_f1 = (2 * raw_precision * raw_recall / (raw_precision + raw_recall)
          if (raw_precision + raw_recall) > 0 else 0.0)

baseline_accuracy = (y_test == 0).mean()   # accuracy of predicting survival for every company in
                           # y_test. Leave it unrounded: a rounded copy will not match.

# Your comment here: the accuracy and the recall, side by side, in words.

# Precision was reported as 0.0: the model flagged 3 companies, all healthy. The model is accurate 99 percent of the time, however, it was unable to flag any of the actual failures. As such if this model picked survive for each company it would be as effective in practice.

# Print the accuracy beside the baseline, then precision, recall and F1, then the counts.

print("accuracy:", raw_accuracy, "baseline:", baseline_accuracy)
print("precision:", raw_precision, "recall:", raw_recall, "f1:", raw_f1)
print("tn fp fn tp:", raw_tn, raw_fp, raw_fn, raw_tp)

accuracy: 0.9899060563661803 baseline: 0.9902058764741155
precision: 0.0 recall: 0.0 f1: 0.0
tn fp fn tp: 9905 3 98 0


## Step 10. Why that result is hollow

Write two or three sentences, in your own words, in `RAW_REFLECTION`.

The accuracy you just printed is high. Say what buys it. Say what the model would be worth to a
lender who wanted to know which of next year's borrowers will not survive, and compare the accuracy
with the baseline that fits no model at all.

In [ ]:
# [YOU COMPLETE THIS] Step 10. Your reading of the raw result.

RAW_REFLECTION = "As indicated in the previous step, failures are rare, around 1%; the high accuracy score reflects that reality. Whereas precision asks how many of the tagged companies actually failed, in this instance, none did. The model missed 98 companies that actually failed, so it provides no predictive utility and would be worth nothing to a lender. Moreover, its accuracy of 98.9% is even slightly below the 99.02% a rule predicting survival for every company achieves with no model at all. "   # two or three sentences in your own words

# Print it.

## Step 11. Rebalance the training set, and only the training set

The failures are rare, so the model has learned that guessing survival is nearly always right. One
standard answer is to change what the model is trained on: copy the rare failing rows until the
training set carries as many failures as survivals. That is **random oversampling**, and
`sklearn.utils.resample` does it with the stock library.

Two rules, and the marking script checks both.

- **Oversample after the split.** Do it before, and copies of the same row land in both halves. The
  model is then tested on rows it was trained on, which is leakage of exactly the kind Step 7 avoids.
- **Leave the test set alone.** The test years must keep their natural, low failure rate, because
  that is the world the model will be used in. A balanced test set answers a question nobody asked.

Produce: `X_train_bal`, `y_train_bal`, `model_bal`. Fit `model_bal` as a fresh pipeline of the same
three steps.

**Comment required.** Say what would go wrong if you oversampled before splitting.

In [ ]:
# [YOU COMPLETE THIS] Step 11. Oversample the failures in the training half only.

train_rows = pd.DataFrame(X_train, columns=RATIO_COLUMNS)
train_rows["target"] = np.asarray(y_train)
majority = train_rows[train_rows["target"] == 0]
minority = train_rows[train_rows["target"] == 1]

minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=RANDOM_STATE,
)   # resample the minority with replacement,
                             # n_samples=len(majority), random_state=RANDOM_STATE
balanced = pd.concat([majority, minority_upsampled])
X_train_bal = balanced[RATIO_COLUMNS]
y_train_bal = balanced["target"]

model_bal = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
model_bal.fit(X_train_bal, y_train_bal)   # a fresh pipeline of the same three steps, with the same two
                   # settings on the logistic regression, fitted on the balanced rows

# Your comment here: what goes wrong if the oversampling happens before the split?
# If we oversample before the split, leakage occurs as copies of the failures from the test split are incorporated causing the model to peek at information it shouldn't have access to.

# Print the size and failure count of the balanced training set, and of the test set.

print("balanced train:", len(X_train_bal), "failures:", int(y_train_bal.sum()))
print("test:", len(X_test), "failures:", int(y_test.sum()))

balanced train: 108548 failures: 54274
test: 10006 failures: 98


## Step 12. Score the rebalanced model, and compare

Score `model_bal` on the same held-out years, at the same `THRESHOLD_A`. Then compare the two sets of
figures.

Produce: `prob_bal`, `pred_bal`, `bal_tn`, `bal_fp`, `bal_fn`, `bal_tp`, `bal_accuracy`,
`bal_precision`, `bal_recall`, `bal_f1`, and `COMPARISON`.

`COMPARISON` is two or three sentences. Name which measures rose and which fell, and say which of the
two models you would give the lender, and why, given that its headline accuracy is the lower of the
two.

In [ ]:
# [YOU COMPLETE THIS] Step 12. Score the rebalanced model and set the two results side by side.

prob_bal = model_bal.predict_proba(X_test)[:, 1]   # as in Step 9: the probabilities, kept unrounded
pred_bal = (prob_bal >= THRESHOLD_A).astype(int)   # as in Step 9: 1 at or above THRESHOLD_A, else 0
bal_tn, bal_fp, bal_fn, bal_tp = confusion_matrix(y_test, pred_bal).ravel()
bal_accuracy = (bal_tp + bal_tn) / len(y_test)
bal_precision = bal_tp / (bal_tp + bal_fp) if (bal_tp + bal_fp) > 0 else 0.0
bal_recall = bal_tp / (bal_tp + bal_fn)
bal_f1 = (2 * bal_precision * bal_recall / (bal_precision + bal_recall)
          if (bal_precision + bal_recall) > 0 else 0.0)

COMPARISON = "We see that accuracy dropped (.99 to.73); however, precision, recall and F1 all increased. As such, the balanced model would be best placed to be used in industry, as it was able to catch 82 of the failures, whereas the original model caught none.   "   # two or three sentences: what rose, what fell, which model, and why


# Print the two sets of measures side by side, then COMPARISON.

print("            raw      rebalanced")
print("accuracy  ", round(raw_accuracy, 4), "  ", round(bal_accuracy, 4))
print("precision ", round(raw_precision, 4), "     ", round(bal_precision, 4))
print("recall    ", round(raw_recall, 4), "     ", round(bal_recall, 4))
print("f1        ", round(raw_f1, 4), "     ", round(bal_f1, 4))
print("tn fp fn tp (bal):", bal_tn, bal_fp, bal_fn, bal_tp)
print(COMPARISON)

            raw      rebalanced
accuracy   0.9899    0.7284
precision  0.0       0.0295
recall     0.0       0.8367
f1         0.0       0.0569
tn fp fn tp (bal): 7206 2702 16 82



## Step 13. Turn the errors into dollars

A missed failure and a false alarm do not cost the same, so the threshold is a business decision
rather than a model setting. The locked cell gives you both figures: `COST_MISSED_FAILURE` for a
company that fails without being flagged, and `COST_FALSE_ALARM` for a healthy company that is
refused.

The cost of an arrangement is the number of missed failures times `COST_MISSED_FAILURE`, plus the
number of false alarms times `COST_FALSE_ALARM`. Nothing else is priced.

Cost three arrangements on the same held-out years: the raw model, the rebalanced model at
`THRESHOLD_A`, and the rebalanced model at `THRESHOLD_B`. Then name the cheaper of the two
thresholds.

Produce: `cost_raw_model`, `cost_bal_050`, `cost_bal_070`, `better_threshold`.

**Comment required.** Say which of the two thresholds flags more companies, and which one costs
the lender less.

In [ ]:
# [YOU COMPLETE THIS] Step 13. Price the mistakes.

cost_raw_model = raw_fn * COST_MISSED_FAILURE + raw_fp * COST_FALSE_ALARM
cost_bal_050 = bal_fn * COST_MISSED_FAILURE + bal_fp * COST_FALSE_ALARM


# Predict again at THRESHOLD_B, count the mistakes, and price them.
pred_bal_070 = (prob_bal >= THRESHOLD_B).astype(int)
tn_070, fp_070, fn_070, tp_070 = confusion_matrix(y_test, pred_bal_070).ravel()
cost_bal_070 = fn_070 * COST_MISSED_FAILURE + fp_070 * COST_FALSE_ALARM

better_threshold = THRESHOLD_A if cost_bal_050 <= cost_bal_070 else THRESHOLD_B
   # the number, THRESHOLD_A or THRESHOLD_B, whichever costs
                        # less; where they cost the same, THRESHOLD_A

# Your comment here: which threshold flags more companies, and which costs less?
# # At a threshold of 0.50 we flag more companies, so we catch more failures but also raise more false alarms. Because a missed failure costs 100 times a false alarm, flagging more readily is cheaper: 0.50 costs about $1.08B against $1.66B at 0.70.

# Print the three costs and the better threshold.

print("raw model:       ", cost_raw_model)
print("rebalanced 0.50: ", cost_bal_050)
print("rebalanced 0.70: ", cost_bal_070)
print("better threshold:", better_threshold)

raw model:        2450750000
rebalanced 0.50:  1075500000
rebalanced 0.70:  1656000000
better threshold: 0.5


## Step 14. What you would tell the lender

Write `INTERPRETATION`: four or five sentences for someone who will not read your code.

Say what the model does, and how well it does it in terms a lender cares about rather than in terms
of accuracy. Say what it costs to run at your chosen threshold, against having no model at all. Then
name one thing you would want before letting it decide anything on its own.

In [62]:
# [YOU COMPLETE THIS] Step 14. The recommendation.

INTERPRETATION = ""   # four or five sentences for a reader who will not read your code

# Print it.

## The summary

Nothing to do here. This cell prints what you produced, so that you can read your own results in one
place before you submit. The marking script reads the variables themselves, not this printout.

(c) 2026 Adeel Mahmood.  All rights reserved.

In [63]:
# [LOCKED] Summary. Do not change anything in this cell.
print("=" * 62)
print("SUBMISSION SUMMARY")
print("=" * 62)
print("name           ", STUDENT_NAME)
print("student number ", STUDENT_NUMBER)
print("version        ", VERSION)
print("rows           ", n_rows)
print("company hash   ", company_set_hash)
print("unreadable     ", market_value_unreadable)
print("failure rate with the flag   ", round(fail_rate_with_flag, 4))
print("failure rate without the flag", round(fail_rate_without_flag, 4))
print("train years    ", train_years[0], "to", train_years[-1])
print("test years     ", test_years[0], "to", test_years[-1])
print("raw        acc", round(raw_accuracy, 4), "rec", round(raw_recall, 4))
print("rebalanced acc", round(bal_accuracy, 4), "rec", round(bal_recall, 4))
print("better threshold", better_threshold)
print("=" * 62)

SUBMISSION SUMMARY
name            Martin Ramirez
student number  001223661
version         1
rows            64686
company hash    aa2675df46f0c0bd8d0de4f30eebba902e97802992522208759038155e41551c
unreadable      31507
failure rate with the flag    0.422
failure rate without the flag 0.0006


TypeError: 'NoneType' object is not subscriptable